In [3]:
import os, re, gzip
import pandas as pd
from pathlib import Path

In [12]:
# --- Helpers ---
def parse_fasta(path):
    """Simple FASTA parser that returns a DataFrame: accession, description, sequence, seqlen."""
    open_fn = gzip.open if str(path).endswith(".gz") else open
    accs, descs, seqs = [], [], []
    with open_fn(path, "rt") as fh:
        acc, desc, seq = None, None, []
        for line in fh:
            if not line:
                continue
            if line.startswith(">"):
                if acc is not None:
                    seqs.append("".join(seq))
                header = line[1:].strip()
                parts  = header.split(None, 1)
                acc    = parts[0]
                desc   = parts[1] if len(parts) > 1 else ""
                accs.append(acc); descs.append(desc); seq = []
            else:
                seq.append(line.strip())
        if acc is not None:
            seqs.append("".join(seq))
    df = pd.DataFrame({"protein_accession": accs, "fasta_description": descs, "sequence": seqs})
    df["sequence_len_fasta"] = df["sequence"].str.len()
    return df


In [ ]:
# --- CONFIG: set these paths ---
IPR_TSV   = "../data/annotation/results_node1/allipr001223_bact_arch_jackseq_fulllength.fasta.tsv"   # <-- your InterProScan TSV
FASTA     = "../data/annotation/results_node1/allipr001223_bact_arch_jackseq_fulllength.fasta"
OUT_DIR   = "../data/annotation/results_node1/SI"



Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# --- Helpers ---
def parse_fasta(path):
    """Simple FASTA parser that returns a DataFrame: accession, description, sequence, seqlen."""
    open_fn = gzip.open if str(path).endswith(".gz") else open
    accs, descs, seqs = [], [], []
    with open_fn(path, "rt") as fh:
        acc, desc, seq = None, None, []
        for line in fh:
            if not line:
                continue
            if line.startswith(">"):
                if acc is not None:
                    seqs.append("".join(seq))
                header = line[1:].strip()
                parts  = header.split(None, 1)
                acc    = parts[0]
                desc   = parts[1] if len(parts) > 1 else ""
                accs.append(acc); descs.append(desc); seq = []
            else:
                seq.append(line.strip())
        if acc is not None:
            seqs.append("".join(seq))
    df = pd.DataFrame({"protein_accession": accs, "fasta_description": descs, "sequence": seqs})
    df["sequence_len_fasta"] = df["sequence"].str.len()
    return df

# --- 1) Load InterProScan TSV (i6/i5 standard 15-col format) ---
# Columns (i5/i6 TSV): protein_accession, sequence_md5, sequence_length, analysis, signature_accession,
# signature_description, start, end, score, status, date, interpro_accession, interpro_description, GO, pathways
cols = [
    "protein_accession","sequence_md5","sequence_length","analysis","signature_accession",
    "signature_description","start","end","score","status","date",
    "interpro_accession","interpro_description","go_annotations","pathway_annotations"
]

ipr = pd.read_csv(
    IPR_TSV,
    sep="\t",
    header=None,
    names=cols,
    dtype=str,
    comment="#",
    na_values=["-", "null", "NULL", ""],
    keep_default_na=False,  # keep empty strings distinct, then set NA above
    engine="python"
)

# Coerce numeric-ish fields
for c in ["sequence_length","start","end"]:
    ipr[c] = pd.to_numeric(ipr[c], errors="coerce")
ipr["score"] = pd.to_numeric(ipr["score"], errors="coerce")

# --- 2) Load FASTA and merge ---
FASTA     = "../../250710/downloads/allipr001223_bact_arch_jackseq_fulllength.fasta"
fa = parse_fasta(FASTA)
merged = ipr.merge(fa, on="protein_accession", how="left")

#remove all rows with "PROSITE patterns" in analysis column
merged = merged[merged["analysis"] != "PROSITE patterns"]

# --- 3) Flag chitinase-related hits -------------------------------------------
import re

# Keywords & patterns (case-insensitive)
terms = [
    r"\bchitinase\b",
    r"\bendochitinase\b",
    r"\bgh18\b",
    r"glycosyl\s+hydrolase",
    r"glycoside\s+hydrolase",
    r"\(trans\)glycosidases?",
    r"\bglycosidases\b",
]

# Hints that imply GH18/chitinase  (NOTE: removed PF08305, PF16875, SM0076/SM00776)
accession_hints = [
    r"\bPF00704\b",
    r"\bIPR001223\b",
    r"\bIPR011583\b",
    r"\bIPR017853\b",
    r"\bIPR050314\b",
    r"\bPTHR11177\b",
    r"\bPTHR45708\b",
    r"\bPS01095\b",
    r"\bPS51910\b",
    r"\bSM00636\b",
    r"\bSSF51445\b",
    r"\bG3DSA:3.20.20.80\b",
    r"\bcd06548\b",
    r"\bPIRSR001103-1\b",
    r"\bcd02871\b",
    r"\bcd06543\b",
    r"\bNF045482\b",
    r"\bPIRSF001103\b",
    r"\bPF23916\b",
    r"\bNF045481\b",
    r"\bPF17801\b",
    r"\bG3DSA:3.20.20.70\b",
    r"\bcd06549\b",
    r"\bPF01522",
    r"\bG3DSA:2.60.40.10:FF:001114\b",
    r"\bG3DSA:3.20.20.80:FF:000153\b",
    r"\bG3DSA:3.10.50.10:FF:000010\b"
]

# Compile regex
kw_pattern = re.compile("(" + "|".join(terms + accession_hints) + ")", flags=re.IGNORECASE)

# Deny-list: always NON-chitinase regardless of other matches
NON_CHIT_SIGS = {"cd00413","PS51762","PR00743","PF07335","PF02838","PF08305", "PF16875", "SM0076", "SM00776","SM00089","cd12215","cd12204","cd04080","cd04081", "G3DSA:2.60.40.10:FF:001114","PF00295","cd20174","PF08329", "cd02848","PS51762","PF00722","PF01120",}

def any_hit(row):
    hay = " | ".join([
        str(row.get("analysis","")),
        str(row.get("signature_accession","")),
        str(row.get("signature_description","")),
        str(row.get("interpro_accession","")),
        str(row.get("interpro_description","")),
    ])
    m = kw_pattern.findall(hay)
    return pd.Series({
        "is_chitinase_related": bool(m),
        "matched_terms": ";".join(sorted(set([x.lower() for x in m]))) if m else ""
    })

flagged = merged.join(merged.apply(any_hit, axis=1))

# ---- OVERRIDE: force deny-list signatures/descriptions to NON-chitinase -------
# Keep this AFTER you computed `flagged = merged.join(...)` and BEFORE you
# construct chitinase_rows.

# Use UPPERCASE for accessions so comparison is consistent
NON_CHIT_SIGS = {
    "PF08305", "PF16875", "SM0076", "SM00776", "SM00089",
    "CD11576",          # GH99-like
    "CD12204",          # Cellulose-binding domain, chitinase-related proteins
    "CD12215",          # Chitin-binding domain of chitinase C
    "CD04080",          # CBM6 (CDD)
    "CD04081",
    "G3DSA:2.60.40.10:FF:001114",
    "PF00295",
    "CD20174",
    "PF08329",
    "G3DSA:2.60.120.180",
    "PR00735",
    "PS00812",
    "PS00592",
    "PR00735",
    "PF01270",
    "PS00698", # GH28
    "PF00759",
    "PS52008",
    "PF00182", 
    "PF17652",
    "PF17801",
    "PF17829",
    "PF18683",
    "SSF48208",
    "G3DSA:3.30.20.10",
    "PF07335",
    "cd00413",
    "PS51762",
    "PR00743",
    "PF02838",
    "PF00722"
    

     # GH28
             # CBM35 (CDD)
}

sig_clean = flagged["signature_accession"].astype(str).str.strip().str.upper()
deny_by_sig = sig_clean.isin(NON_CHIT_SIGS)

# Be explicit: make description lowercase and use regex=False so parentheses etc. are literal
desc_clean = flagged["signature_description"].astype(str).str.lower()
deny_by_desc = (
    desc_clean.str.contains("glycoside hydrolase family 99", regex=False, na=False) |
    desc_clean.str.contains("cellulose-binding domain",        regex=False, na=False) |
    desc_clean.str.contains("chitin-binding domain of chitinase c", regex=False, na=False) |
    desc_clean.str.contains("carbohydrate binding module 6",   regex=False, na=False) |
    desc_clean.str.contains("cbm6",                            regex=False, na=False) |
    desc_clean.str.contains("carbohydrate binding module family 35", regex=False, na=False) |
    desc_clean.str.contains("cbm35",                           regex=False, na=False) |
    desc_clean.str.contains("chitinase a1", regex=False, na=False) |
    desc_clean.str.contains("glycosyl hydrolases family 28", regex=False, na=False) |
    desc_clean.str.contains("an unknown function domain of Listeria innocua LinChi78 GH18 chitinase that is essential for its catalytic activity; found in similar chitinase-like proteins", regex=False, na=False) |
    desc_clean.str.contains("chitinase a, n-terminal domain", regex=False, na=False) |
    desc_clean.str.contains("glycoside hydrolase family 11/12", regex=False, na=False) |
    desc_clean.str.contains("glycoside hydrolase family 19", regex=False, na=False) |
    desc_clean.str.contains("glycoside hydrolase family 16", regex=False, na=False) |
    desc_clean.str.contains("n-terminal early set domain associated with the catalytic domain of chitinase", regex=False, na=False) |
    desc_clean.str.contains("glycoside hydrolase, family 8", regex=False, na=False) |
    desc_clean.str.contains("glycosyl hydrolases family 8 signature", regex=False, na=False) |
    desc_clean.str.contains("glycoside hydrolase family 8", regex=False, na=False) |
    desc_clean.str.contains("glycosyl hydrolase family 9", regex=False, na=False) |
    desc_clean.str.contains("Glycosyl hydrolases family 81 (GH81) domain profile", regex=False, na=False) |
    desc_clean.str.contains("chitinase class i", regex=False, na=False) |
    desc_clean.str.contains("glycosyl hydrolase family 81 c-terminal domain", regex=False, na=False) |
    desc_clean.str.contains("alpha galactosidase c-terminal beta sandwich domain", regex=False, na=False) |
    desc_clean.str.contains("glycosyl hydrolase family 115 c-terminal domain", regex=False, na=False) |
    desc_clean.str.contains("chitinase w immunoglobulin-like domain", regex=False, na=False) |
    desc_clean.str.contains("six-hairpin glycosidases", regex=False, na=False) |
    desc_clean.str.contains("endochitinase, domain 2", regex=False, na=False)
)

final_deny = deny_by_sig | deny_by_desc

if final_deny.any():
    flagged.loc[final_deny, "is_chitinase_related"] = False
    flagged.loc[final_deny, "matched_terms"] = ""
    print(
        f"⚠️ Overrode {final_deny.sum()} rows to non-chitinase "
        f"(by accession: {deny_by_sig.sum()}, by description: {deny_by_desc.sum()})"
    )

# Continue as before:
chitinase_proteins = (
    flagged.loc[flagged["is_chitinase_related"], ["protein_accession"]]
    .drop_duplicates()
    .sort_values("protein_accession")
)

chitinase_signatures = (
    flagged.loc[flagged["is_chitinase_related"],
                ["analysis","signature_accession","signature_description",
                 "interpro_accession","interpro_description"]]
    .drop_duplicates()
    .sort_values(["analysis","signature_accession"])
)

chitinase_rows = flagged.loc[flagged["is_chitinase_related"]].copy()


# --- 4) Useful summaries -------------------------------------------------------
# A) Unique proteins that have at least one chitinase-related hit
chitinase_proteins = (
    flagged.loc[flagged["is_chitinase_related"], ["protein_accession"]]
    .drop_duplicates()
    .sort_values("protein_accession")
)

# B) Unique chitinase-related signatures/accessions (what “hit”)
chitinase_signatures = (
    flagged.loc[flagged["is_chitinase_related"],
                ["analysis","signature_accession","signature_description","interpro_accession","interpro_description"]]
    .drop_duplicates()
    .sort_values(["analysis","signature_accession"])
)

# C) Full filtered table (only rows that matched)
chitinase_rows = flagged.loc[flagged["is_chitinase_related"]].copy()

# --- Find sequences from FASTA not appearing in any chitinase-related hit ---
chitinase_ids = set(chitinase_rows["protein_accession"])
fasta_ids = set(fa["protein_accession"])

missing = sorted(fasta_ids - chitinase_ids)
print(f"Total FASTA sequences: {len(fasta_ids)}")
print(f"Chitinase-related sequences: {len(chitinase_ids)}")
print(f"Sequences with NO chitinase-related hits: {len(missing)}")

# optional: view and/or save them
missing_df = fa[fa["protein_accession"].isin(missing)].copy()
display(missing_df.head())

missing_df.to_csv(
    os.path.join(OUT_DIR, "non_chitinase_sequences.tsv"),
    sep="\t", index=False
)


# --- 5) Save outputs -----------------------------------------------------------
flagged_out   = os.path.join(OUT_DIR, "interproscan_merged_with_fasta.tsv")
protlist_out  = os.path.join(OUT_DIR, "chitinase_related_proteins.tsv")
sigs_out      = os.path.join(OUT_DIR, "chitinase_related_signatures.tsv")
rows_out      = os.path.join(OUT_DIR, "interproscan_chitinase_rows.tsv")

flagged.to_csv(flagged_out, sep="\t", index=False)
chitinase_proteins.to_csv(protlist_out, sep="\t", index=False)
chitinase_signatures.to_csv(sigs_out, sep="\t", index=False)
chitinase_rows.to_csv(rows_out, sep="\t", index=False)

print("✓ Wrote:")
print("  -", flagged_out)
print("  -", protlist_out)
print("  -", sigs_out)
print("  -", rows_out)

# --- 6) Quick peek: top 10 chitinase-related hits
display(chitinase_rows.head(10))


⚠️ Overrode 14219 rows to non-chitinase (by accession: 12001, by description: 11378)
Total FASTA sequences: 39582
Chitinase-related sequences: 39582
Sequences with NO chitinase-related hits: 0


,protein_accession,fasta_description,sequence,sequence_len_fasta


✓ Wrote:
  - SI/interproscan_merged_with_fasta.tsv
  - SI/chitinase_related_proteins.tsv
  - SI/chitinase_related_signatures.tsv
  - SI/interproscan_chitinase_rows.tsv


,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,date,interpro_accession,interpro_description,go_annotations,pathway_annotations,fasta_description,sequence,sequence_len_fasta,is_chitinase_related,matched_terms
0,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,CATH-Gene3D,G3DSA:3.10.50.10,NaN,279,334,8.600000e-85,T,20-10-2025,IPR029070,Chitinase insertion domain superfamily,NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,chitinase
1,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,CATH-Gene3D,G3DSA:3.20.20.80,Glycosidases,85,368,8.600000e-85,T,20-10-2025,NaN,NaN,NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,g3dsa:3.20.20.80;glycosidases
2,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,CDD,cd02874,NaN,59,370,NaN,T,20-10-2025,IPR041704,"CFLE, GH18 catalytic domain",NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,gh18
4,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,PANTHER,PTHR46066,CHITINASE DOMAIN-CONTAINING PROTEIN 1 FAMILY M...,81,369,NaN,T,20-10-2025,NaN,NaN,NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,chitinase
5,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,58,378,3.759639e+01,T,20-10-2025,IPR001223,"Glycoside hydrolase family 18, catalytic domain",NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,gh18;glycoside hydrolase;glycosyl hydrolase;ip...
7,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,Pfam,PF00704,Glycosyl hydrolases family 18,131,362,1.700000e-30,T,20-10-2025,IPR001223,"Glycoside hydrolase family 18, catalytic domain",NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...
8,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,SMART,SM00636,NaN,58,363,7.700000e-19,T,20-10-2025,IPR011583,"Chitinase II/V-like, catalytic domain",NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,chitinase;ipr011583;sm00636
9,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,SUPERFAMILY,SSF51445,(Trans)glycosidases,59,376,NaN,T,20-10-2025,IPR017853,Glycoside hydrolase superfamily,NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,True,(trans)glycosidases;glycoside hydrolase;ipr017...
10,A9ZR80,F0AA72F2E31F01A21BA84CAFEAF8978B,115,CATH-Gene3D,G3DSA:3.20.20.80,Glycosidases,2,115,1.200000e-33,T,20-10-2025,NaN,NaN,NaN,NaN,,VVGGLQPGQPRDAHNYVLLLTELRRAIGPNKLLTIAVGASPRDIEP...,115,True,g3dsa:3.20.20.80;glycosidases
11,A9ZR80,F0AA72F2E31F01A21BA84CAFEAF8978B,115,PANTHER,PTHR11177,CHITINASE,7,115,NaN,T,20-10-2025,IPR050314,Glycosyl Hydrolase Family 18,NaN,NaN,,VVGGLQPGQPRDAHNYVLLLTELRRAIGPNKLLTIAVGASPRDIEP...,115,True,chitinase;glycosyl hydrolase;ipr050314;pthr11177


In [2]:
# --- 3b) Also build the inverse rows (non-chitinase rows) ----------------------
non_chitinase_rows = flagged.loc[~flagged["is_chitinase_related"]].copy()

# Optional summaries for the inverse set
non_chitinase_proteins = (
    non_chitinase_rows[["protein_accession"]]
    .drop_duplicates()
    .sort_values("protein_accession")
)

# --- 5b) Save inverse outputs --------------------------------------------------
non_rows_out   = os.path.join(OUT_DIR, "interproscan_non_chitinase_rows.tsv")
non_protlist_out = os.path.join(OUT_DIR, "non_chitinase_proteins.tsv")

non_chitinase_rows.to_csv(non_rows_out, sep="\t", index=False)
non_chitinase_proteins.to_csv(non_protlist_out, sep="\t", index=False)

print("  -", non_rows_out)
print("  -", non_protlist_out)

# Quick peek at the inverse
display(non_chitinase_rows.head(10))

  - SI/interproscan_non_chitinase_rows.tsv
  - SI/non_chitinase_proteins.tsv


,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,date,interpro_accession,interpro_description,go_annotations,pathway_annotations,fasta_description,sequence,sequence_len_fasta,is_chitinase_related,matched_terms
3,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,MobiDB-lite,mobidb-lite,Consensus disorder prediction,22,47,NaN,T,20-10-2025,NaN,NaN,NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,False,
6,A0A0J1IBL6,F0A6A68C835E0C0C47B7D81A5D5BCDF8,382,PROSITE profiles,PS51257,Prokaryotic membrane lipoprotein lipid attachm...,1,21,6.000000e+00,T,20-10-2025,NaN,NaN,NaN,NaN,,MKRFLTLIVISSLMLVSMAGCGSVPQSPQPQQPQKMTDAGQPPQAE...,382,False,
15,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,CATH-Gene3D,G3DSA:2.60.120.260,Galactose-binding domain-like,30,153,3.600000e-21,T,20-10-2025,NaN,NaN,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
16,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,CATH-Gene3D,G3DSA:2.60.40.10,Immunoglobulins,161,244,7.500000e-19,T,20-10-2025,IPR013783,Immunoglobulin-like fold,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
18,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,CDD,cd00063,NaN,174,244,NaN,T,20-10-2025,IPR003961,Fibronectin type III,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
20,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,MobiDB-lite,mobidb-lite,Consensus disorder prediction,152,171,NaN,T,20-10-2025,NaN,NaN,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
24,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,PROSITE profiles,PS50853,Fibronectin type-III domain profile,165,247,1.480891e+01,T,20-10-2025,IPR003961,Fibronectin type III,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
26,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,Pfam,PF00041,Fibronectin type III domain,165,242,2.600000e-06,T,20-10-2025,IPR003961,Fibronectin type III,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
28,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,Pfam,PF02018,Carbohydrate binding domain,30,139,3.000000e-09,T,20-10-2025,IPR003305,"Carbohydrate-binding, CenC-like",NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,
29,M2PU94,F0AB0DC4B166CB4D7751E068916E98A4,589,SMART,SM00060,Fibronectin type 3 domain,163,234,9.000000e-10,T,20-10-2025,IPR003961,Fibronectin type III,NaN,NaN,,MRRSRSLAFLAVVVLTFAGLITGGGTASAANLLLNPGFEAGSTSGW...,589,False,


In [5]:
non_chitinase_rows = pd.read_csv('SI/interproscan_non_chitinase_rows.tsv', sep="\t")
non_chitinase_signatures = (
    non_chitinase_rows[["analysis",
                        "signature_accession",
                        "signature_description",
                        "interpro_accession",
                        "interpro_description"]]
    .drop_duplicates()
    .sort_values(["analysis","signature_accession"])
)

non_chitinase_signatures

,analysis,signature_accession,signature_description,interpro_accession,interpro_description
54488,CATH-FunFam,G3DSA:1.10.1330.10:FF:000001,Endoglucanase D,NaN,NaN
201589,CATH-FunFam,G3DSA:1.20.1440.230:FF:000001,Mitochondrial NADH dehydrogenase flavoprotein 1,NaN,NaN
193436,CATH-FunFam,G3DSA:1.50.10.10:FF:000020,Endoglucanase,NaN,NaN
1326,CATH-FunFam,G3DSA:2.10.10.20:FF:000004,Peptidase M28,NaN,NaN
140806,CATH-FunFam,G3DSA:2.10.10.20:FF:000005,Chitodextrinase,NaN,NaN
...,...,...,...,...,...
28256,SUPERFAMILY,SSF75011,"3-carboxy-cis,cis-mucoante lactonizing enzyme",NaN,NaN
474,SUPERFAMILY,SSF81296,E set domains,IPR014756,Immunoglobulin E-set
225647,SUPERFAMILY,SSF81995,beta-sandwich domain of Sec23/24,NaN,NaN
50188,SUPERFAMILY,SSF89260,Collagen-binding domain,NaN,NaN


In [8]:
top_signature = (
    non_chitinase_rows["signature_accession"]
    .value_counts()
    .reset_index()
)
top_signature.columns = ["signature_accession", "count"]
top_signature.head(15)

,signature_accession,count
0,mobidb-lite,25635
1,G3DSA:2.60.40.10,14948
2,SM00495,9913
3,G3DSA:2.10.10.20,8910
4,SSF51055,8731
5,SM00060,7147
6,cd00063,7069
7,PS50853,7010
8,PF00041,6556
9,cd12215,6508


In [ ]:
# --- Frequency summary of chitinase-related hits (robust version) ---
import pandas as pd

# 1) Most common analysis type
if "analysis" not in chitinase_rows.columns:
    raise KeyError("Column 'analysis' not found in chitinase_rows — check your TSV parser cell!")

top_analysis = (
    chitinase_rows["analysis"]
    .value_counts()
    .reset_index()
)
top_analysis.columns = ["analysis", "count"]

print("🔹 Top analyses contributing to chitinase-related hits:")
display(top_analysis.head(10))

most_common_analysis = top_analysis.loc[0, "analysis"]
print(f"\n✅ Most common analysis: {most_common_analysis}")

# 2) Most common signature accession
if "signature_accession" not in non_chitinase_rows.columns:
    raise KeyError("Column 'signature_accession' not found in chitinase_rows — check your TSV parser cell!")

top_signature = (
    non_chitinase_rows["signature_accession"]
    .value_counts()
    .reset_index()
)
top_signature.columns = ["signature_accession", "count"]

print("\n🔹 Top signature_accession values among chitinase-related hits:")
display(top_signature.head(10))

most_common_signature = top_signature.loc[0, "signature_accession"]
print(f"\n✅ Most common signature accession: {most_common_signature}")

# 3) Optional: details for that signature
sig_subset = (
    chitinase_rows.loc[
        chitinase_rows["signature_accession"] == most_common_signature,
        ["analysis", "signature_accession", "signature_description", "interpro_accession", "interpro_description"]
    ]
    .drop_duplicates()
)
print("\n🔹 Details of most common signature:")
display(sig_subset)


🔹 Top analyses contributing to chitinase-related hits:


,analysis,count
0,CATH-Gene3D,67439
1,SUPERFAMILY,53242
2,PROSITE profiles,39037
3,Pfam,38201
4,PANTHER,35547
5,SMART,31254
6,CDD,22666
7,PIRSR,13287
8,CATH-FunFam,2741
9,NCBIFAM,587



✅ Most common analysis: CATH-Gene3D

🔹 Top signature_accession values among chitinase-related hits:


,signature_accession,count
0,G3DSA:3.20.20.80,40713
1,SSF51445,40009
2,PS51910,39036
3,PF00704,36361
4,SM00636,31254
5,G3DSA:3.10.50.10,23653
6,PTHR11177,16418
7,PIRSR001103-1,13287
8,SSF54556,11782
9,cd06548,11059



✅ Most common signature accession: G3DSA:3.20.20.80

🔹 Details of most common signature:


,analysis,signature_accession,signature_description,interpro_accession,interpro_description
1,CATH-Gene3D,G3DSA:3.20.20.80,Glycosidases,NaN,NaN


In [9]:
#a chitinase ID fingerprint dataframe
chitinase_rows = pd.read_csv('SI/interproscan_chitinase_rows.tsv', sep="\t")
import pandas as pd

# --- Base preparation ---
tmp = chitinase_rows.copy()
tmp["start_num"] = pd.to_numeric(tmp["start"], errors="coerce")
tmp = tmp.sort_values(["protein_accession", "start_num"], kind="mergesort")

# --- Helper: ordered unique accessions ---
def ordered_unique(series: pd.Series) -> list[str]:
    seen = set()
    out = []
    for v in series.dropna().astype(str):
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

# --- Build main and rare fingerprints ---
def build_fingerprints(g: pd.DataFrame) -> pd.Series:
    sigs = ordered_unique(g["signature_accession"])
    # Main fingerprint
    fprint = " ".join(sigs)
    
    # Rare fingerprint (exclude core GH18 markers)
    exclude = {
        "G3DSA:3.20.20.80",
        "SSF51445",
        "PS51910",
        "PF00704",
        "SM00636",
        "G3DSA:3.10.50.10",
    }
    rare = [s for s in sigs if s not in exclude]
    rare_fprint = " ".join(rare)
    
    # CID domain presence (G3DSA:3.10.50.10)
    cid_domain = any(s == "G3DSA:3.10.50.10" for s in sigs)
    
    return pd.Series({
        "chitinase_fingerprint": fprint,
        "rare_chitinase_ID_fprint": rare_fprint,
        "CID_domain": cid_domain,
    })

chitinase_fingerprints = (
    tmp.groupby("protein_accession", sort=False)
       .apply(build_fingerprints)
       .reset_index()
)

# --- Add count of unique signatures ---
chitinase_fingerprints["num_unique_chitinase_signatures"] = (
    chitinase_fingerprints["chitinase_fingerprint"]
    .str.split()
    .apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
)

#display(chitinase_fingerprints.head(10))

# Optional: save
chitinase_fingerprints.to_csv("SI/chitinase_fingerprints_with_rare.tsv", sep="\t", index=False)
chitinase_fingerprints

,protein_accession,chitinase_fingerprint,rare_chitinase_ID_fprint,CID_domain,num_unique_chitinase_signatures
0,A0A010YG22,G3DSA:3.20.20.80 cd06543 PTHR42976 PS51910 SSF...,cd06543 PTHR42976,False,5
1,A0A011PM39,PS51910 SM00636 SSF51445 G3DSA:3.20.20.80 PF00...,G3DSA:3.20.20.370 SSF88713 PF01522,True,9
2,A0A011PXQ4,PS51910 SM00636 PF00704 SSF51445 G3DSA:3.20.20...,SSF88713 G3DSA:3.20.20.370 PF01522,True,9
3,A0A014LJM1,PTHR42976 G3DSA:3.20.20.80 PS51910 SSF51445 cd...,PTHR42976 cd06543,False,5
4,A0A014LR51,PTHR45708 G3DSA:3.20.20.80 SSF51445 cd02871 PS...,PTHR45708 cd02871 PIRSR001103-1,False,8
...,...,...,...,...,...
39577,X6Q4Q0,PS51910 SM00636 SSF51445 G3DSA:3.20.20.80 PF00...,G3DSA:3.20.20.370 SSF88713 PF01522,True,9
39578,X8GPF2,PS51910 SM00636 PTHR46066 SSF51445 G3DSA:3.20....,PTHR46066,True,7
39579,X8HE75,PS51910 SM00636 SSF51445 PTHR46066 G3DSA:3.20....,PTHR46066,True,7
39580,Z9JN00,cd02874 PTHR46066 PS51910 SM00636 PF00704 SSF5...,cd02874 PTHR46066,True,8


dedupes gh18 domains per sequence taking into account domai overlap so if  a sequence has more than 1 gh18 domain this is accounted for, gh18 codes are preferenced but if a domain is identified that does not have one of top three identifiers it is maintained, not discarded

In [14]:
# --- GH18-centric clustering with sweep-line (fast) ----------------------------
import pandas as pd
import numpy as np
from pathlib import Path
import os

OUT_DIR = "SI"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# sanity checks
need_cols = ["protein_accession","signature_accession","interpro_accession",
             "start","end","score","analysis"]
missing = [c for c in need_cols if c not in chitinase_rows.columns]
if missing:
    raise KeyError(f"Missing in chitinase_rows: {missing}")
for c in ["protein_accession","sequence","sequence_len_fasta","fasta_description"]:
    if c not in fa.columns:
        raise KeyError(f"Missing in FASTA df: {c}")

# priority & GH18-like tags
PRIORITY = ["PF00704","PS51910", "SSF51445"]
prio_map = {acc:i+1 for i,acc in enumerate(PRIORITY)}
GH18_SIGS = {
    "PS51910","SSF51445","PF00704", "PS01095","SM00636","cd06548",
    "IPR001223","IPR011583","IPR050314","IPR017853","G3DSA:3.20.20.80"
}

# global frequency for fallback
sig_freq_map = chitinase_rows["signature_accession"].value_counts(dropna=False).to_dict()

# numeric copy
df = chitinase_rows.copy()
df["start_num"] = pd.to_numeric(df["start"], errors="coerce")
df["end_num"]   = pd.to_numeric(df["end"], errors="coerce")
df["score_num"] = pd.to_numeric(df["score"], errors="coerce")
df = df.dropna(subset=["protein_accession","start_num","end_num"]).copy()
df["start_num"] = df["start_num"].astype(int)
df["end_num"]   = df["end_num"].astype(int)
df["span"]      = (df["end_num"] - df["start_num"] + 1).clip(lower=0)
df["priority_rank"]   = df["signature_accession"].map(prio_map)
df["sig_freq_global"] = df["signature_accession"].map(sig_freq_map).fillna(0).astype(int)

# --- helpers for % overlap inside non-GH18 clusters ---------------------------
POS_TOL = 5
OVERLAP_FRAC = 0.80  # tweak 0.7–0.9 as you like

def overlap_frac(a_s, a_e, b_s, b_e):
    inter = max(0, min(a_e, b_e) - max(a_s, b_s) + 1)
    shorter = max(1, min(a_e - a_s + 1, b_e - b_s + 1))
    return inter / shorter

def same_domain(a_s, a_e, b_s, b_e):
    # endpoints within tolerance OR ≥% of the shorter interval overlaps
    if abs(a_s - b_s) <= POS_TOL and abs(a_e - b_e) <= POS_TOL:
        return True
    return overlap_frac(a_s, a_e, b_s, b_e) >= OVERLAP_FRAC

def choose_rep_priority_then_freq(cdf: pd.DataFrame) -> pd.Series:
    # GH18 priority first, then global frequency, span↓, score↑, start↑
    if cdf["signature_accession"].isin(PRIORITY).any():
        cand = cdf[cdf["signature_accession"].isin(PRIORITY)].copy()
        cand = cand.sort_values(
            ["priority_rank","span","score_num","start_num"],
            ascending=[True, False, True, True]
        )
        return cand.iloc[0]
    # fallback for non-GH18 sets
    cand = cdf.sort_values(
        ["sig_freq_global","span","score_num","start_num"],
        ascending=[False, False, True, True]
    )
    return cand.iloc[0]


# sweep-line clustering per protein (ANY overlap creates a cluster)
# clusters that contain a GH18 anchor are collapsed to one row; others are kept as-is
kept_parts = []

gh18_like = lambda s, i: (isinstance(s,str) and s in GH18_SIGS) or (isinstance(i,str) and i in GH18_SIGS)

for prot, p in df.groupby("protein_accession", sort=False):
    p = p.sort_values(["start_num","end_num"]).reset_index(drop=True)
    clusters_idx = []
    cur = [0]  # indices for current cluster
    cur_end = p.loc[0, "end_num"]

    for i in range(1, len(p)):
        s = p.loc[i, "start_num"]
        # ANY overlap: new.start <= current cluster end → same cluster
        if s <= cur_end:
            cur.append(i)
            # extend cluster end if needed
            if p.loc[i, "end_num"] > cur_end:
                cur_end = p.loc[i, "end_num"]
        else:
            clusters_idx.append(cur)
            cur = [i]
            cur_end = p.loc[i, "end_num"]
    clusters_idx.append(cur)

    # process clusters
    for idxs in clusters_idx:
        cdf = p.iloc[idxs].copy()

        # does cluster contain a GH18 anchor?
        has_gh18 = (
            cdf["signature_accession"].isin(GH18_SIGS).any() or
            cdf["interpro_accession"].isin(GH18_SIGS).any()
        )

        if has_gh18:
            # Collapse the whole cluster to ONE representative using GH18 priority
            rep = choose_rep_priority_then_freq(cdf)
            kept_parts.append(rep.to_frame().T)
        else:
            # Split this cluster into subcomponents where members are "same domain"
            # under the %overlap rule, then pick ONE rep per subcomponent
            cdf = cdf.reset_index(drop=True)
            n = len(cdf)
            # build graph with edges only when same_domain() is True
            comp_adj = [[] for _ in range(n)]
            s = cdf["start_num"].to_numpy()
            e = cdf["end_num"].to_numpy()
            for i in range(n):
                for j in range(i+1, n):
                    if same_domain(int(s[i]), int(e[i]), int(s[j]), int(e[j])):
                        comp_adj[i].append(j); comp_adj[j].append(i)

            # find connected components and pick a representative in each
            seen = [False]*n
            reps = []
            for i in range(n):
                if seen[i]:
                    continue
                stack = [i]; seen[i] = True; comp_idx = [i]
                while stack:
                    k = stack.pop()
                    for nb in comp_adj[k]:
                        if not seen[nb]:
                            seen[nb] = True
                            stack.append(nb)
                            comp_idx.append(nb)
                sub = cdf.iloc[comp_idx].copy()
                rep = choose_rep_priority_then_freq(sub)
                reps.append(rep)

            kept_parts.append(pd.DataFrame(reps))

# combine and finish
parsed_chitinase_dom = pd.concat(kept_parts, ignore_index=True)

# attach sequences if not present
if "sequence" not in parsed_chitinase_dom.columns:
    parsed_chitinase_dom = parsed_chitinase_dom.merge(
        fa[["protein_accession","sequence","sequence_len_fasta","fasta_description"]],
        on="protein_accession", how="left"
    )

parsed_chitinase_dom = parsed_chitinase_dom.sort_values(
    ["protein_accession","start_num","end_num"]
).reset_index(drop=True)

# (optional) write
out_path = os.path.join(OUT_DIR, "parsed_chitinase_dom.tsv")
parsed_chitinase_dom.to_csv(out_path, sep="\t", index=False)
print(f"✓ Wrote {out_path}   rows={len(parsed_chitinase_dom)}")

display(parsed_chitinase_dom.head(20))


✓ Wrote SI/parsed_chitinase_dom.tsv   rows=41904


,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,...,sequence,sequence_len_fasta,is_chitinase_related,matched_terms,start_num,end_num,score_num,span,priority_rank,sig_freq_global
0,A0A010YG22,BCAC65E13E44ACC0D5889A40D5B1A2F8,354,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,354,10.203516,T,...,MARGQRRRNEKGRWVLIGVAVVAVIAAAATAVVFLRDGDDGKDGTV...,354,True,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,354,10.203516,299,2.0,39036
1,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF00704,Glycosyl hydrolases family 18,137,428,0.0,T,...,MSQPPFLDPAPAPGASPKTAKAVFFDPANKRWPRLRLGMALIGVAL...,1156,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,137,428,0.0,292,1.0,36361
2,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,503,716,0.0,T,...,MSQPPFLDPAPAPGASPKTAKAVFFDPANKRWPRLRLGMALIGVAL...,1156,True,glycoside hydrolase,503,716,0.0,214,NaN,1399
3,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,Pfam,PF00704,Glycosyl hydrolases family 18,136,428,0.0,T,...,MSQSLSGDPQPRPGSGPKIAAAVFFDPANKRWPRLRLGMALIAVTL...,1156,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,136,428,0.0,293,1.0,36361
4,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,504,716,0.0,T,...,MSQSLSGDPQPRPGSGPKIAAAVFFDPANKRWPRLRLGMALIAVTL...,1156,True,glycoside hydrolase,504,716,0.0,213,NaN,1399
5,A0A014LJM1,8495312E26224E3E1925B2F1461C43E6,344,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,344,10.605026,T,...,MNHSEDLSARDRARRRRRRTVLAVAAALAATAASTTAAAWADNTGQ...,344,True,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,344,10.605026,289,2.0,39036
6,A0A014LR51,9F864E95FB4997720FE7F921ECFB6456,336,Pfam,PF00704,Glycosyl hydrolases family 18,51,270,0.0,T,...,MVRAQFTKRLLGAAAVVALAAGGLISAGSANASEEHPDAKQHAAAV...,336,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,51,270,0.0,220,1.0,36361
7,A0A014LVP2,44CFF99BCD9A69393EB542696BBF9637,311,Pfam,PF00704,Glycosyl hydrolases family 18,80,253,0.0,T,...,MFTLVRSRLRTAAFALSAVAALAFAGTAGTAASATTGAAAAPAPAK...,311,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,80,253,0.0,174,1.0,36361
8,A0A014M4I2,26C57647ACF0EF3CEFA8F8E9E6E5C545,421,Pfam,PF00704,Glycosyl hydrolases family 18,57,410,0.0,T,...,MLRPGRLSAALAAICTAVLTATLLAGSAAAGEPRSAGIPQATSKAA...,421,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,57,410,0.0,354,1.0,36361
9,A0A014MIZ7,F369CDD9CFC7DB25C25543325A21CB1C,350,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,341,9.859364,T,...,MSSTHRRKASRTTKVIGATAAAVVAGGTAFALSGAANAGTVSEKTS...,350,True,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,341,9.859364,286,2.0,39036


In [15]:
# --- Order GH18 dataframe and count multi-domain proteins ----------------------
parsed_chitinase_dom = pd.read_csv("SI/parsed_chitinase_dom.tsv", sep="\t")
# Ensure numeric columns for sorting
parsed_chitinase_dom["start"] = pd.to_numeric(parsed_chitinase_dom["start"], errors="coerce")
parsed_chitinase_dom["end"]   = pd.to_numeric(parsed_chitinase_dom["end"], errors="coerce")

#remove rows with Nan in start or end column
parsed_chitinase_dom = parsed_chitinase_dom.dropna(subset=["start", "end"])

# Sort by accession then start position
parsed_chitinase_dom = parsed_chitinase_dom.sort_values(["protein_accession", "start"]).reset_index(drop=True)

# Count how many GH18 domains per accession
domain_counts = parsed_chitinase_dom["protein_accession"].value_counts()
multi_domain_accessions = domain_counts[domain_counts > 1]

print(f"✅ Total proteins with GH18 domain(s): {len(domain_counts)}")
print(f"🔹 Proteins with >1 non-overlapping GH18 domains: {len(multi_domain_accessions)}")

# Optional: view a few examples of multi-domain proteins
if len(multi_domain_accessions) > 0:
    print("\nExamples (accession : domain_count):")
    display(multi_domain_accessions.head(10))

# Save the sorted dataframe
sorted_out = os.path.join(OUT_DIR, "parsed_chitinase_dom_sorted.tsv")
parsed_chitinase_dom.to_csv(sorted_out, sep="\t", index=False)
print(f"\n✓ Sorted and saved to: {sorted_out}")


✅ Total proteins with GH18 domain(s): 39582
🔹 Proteins with >1 non-overlapping GH18 domains: 2261

Examples (accession : domain_count):


protein_accession
A0AAJ4Y936    3
A0A0N1GCJ3    3
A0A2E0AK96    3
A0A941CEH3    3
Q9ZFP7        3
A0A2D9Y0Y6    3
A0A416W5R2    3
A0A3A9VL07    3
A0A080LX25    3
A0A6H1LR75    3
Name: count, dtype: int64


✓ Sorted and saved to: SI/parsed_chitinase_dom_sorted.tsv


non chitinase domain dataset

In [16]:
# --- Frequency of signature_accession in non_chitinase_rows ------------------
import os
import pandas as pd

# Ensure column exists
if "signature_accession" not in non_chitinase_rows.columns:
    raise KeyError("Column 'signature_accession' not found in non_chitinase_rows")

# 1. Count frequency of each signature_accession
sig_freq = (
    non_chitinase_rows["signature_accession"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "signature_accession", "signature_accession": "count"})
)

# Fix column names explicitly to be safe
sig_freq.columns = ["signature_accession", "count"]

# 2. Merge basic info about each signature
info_cols = [
    "signature_accession",
    "analysis",
    "signature_description",
    "interpro_accession",
    "interpro_description",
]
info_cols = [c for c in info_cols if c in non_chitinase_rows.columns]

sig_info = non_chitinase_rows[info_cols].drop_duplicates(subset=["signature_accession"])

sig_summary = (
    sig_info.merge(sig_freq, on="signature_accession", how="left")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

# 3. Display and optionally save
print(f"Total unique signature_accession identifiers: {len(sig_summary)}")
display(sig_summary.head(20))

out_path = os.path.join(OUT_DIR, "non_chitinase_signature_frequencySI.tsv")
sig_summary.to_csv(out_path, sep="\t", index=False)
print(f"✓ Wrote signature frequency summary: {out_path}")


Total unique signature_accession identifiers: 785


,signature_accession,analysis,signature_description,interpro_accession,interpro_description,count
0,mobidb-lite,MobiDB-lite,Consensus disorder prediction,NaN,NaN,25635
1,G3DSA:2.60.40.10,CATH-Gene3D,Immunoglobulins,IPR013783,Immunoglobulin-like fold,14948
2,SM00495,SMART,Chitin-binding domain type 3,IPR003610,Carbohydrate binding module family 5/12,9913
3,G3DSA:2.10.10.20,CATH-Gene3D,Carbohydrate-binding module superfamily 5/12,NaN,NaN,8910
4,SSF51055,SUPERFAMILY,Carbohydrate binding domain,IPR036573,Carbohydrate-binding module superfamily 5/12,8731
5,SM00060,SMART,Fibronectin type 3 domain,IPR003961,Fibronectin type III,7147
6,cd00063,CDD,NaN,IPR003961,Fibronectin type III,7069
7,PS50853,PROSITE profiles,Fibronectin type-III domain profile,IPR003961,Fibronectin type III,7010
8,PF00041,Pfam,Fibronectin type III domain,IPR003961,Fibronectin type III,6556
9,cd12215,CDD,Chitin-binding domain of chitinase C,NaN,NaN,6508


✓ Wrote signature frequency summary: SI/non_chitinase_signature_frequencySI.tsv


In [17]:
# --- Deduplicate by overlapping identifiers per protein, preferring Pfam -------
# Rule per protein:
#   - Build groups of hits that significantly overlap (>= 50% of the shorter interval
#     OR both ends within ±5 aa).
#   - From each group, keep ONE:
#       1) Prefer Pfam hits (analysis == 'Pfam', case-insensitive).
#       2) Else pick the hit with the highest global frequency of signature_accession.
#       3) Tie-break: longer span, then lower numeric score, then earlier start.
#
# Input: non_chitinase_rows (DataFrame with at least: protein_accession, analysis,
#                           signature_accession, start, end, score)
# Output: non_chitinase_dedup_prefpfam (deduplicated DataFrame)

import pandas as pd
import numpy as np
from collections import deque
import os

# ------------------ parameters ------------------
POS_TOL = 5           # positional tolerance in aa for "almost same"
OVERLAP_FRAC = 0.30   # "significant overlap" threshold (fraction of the shorter interval)
OUT_PATH = os.path.join(OUT_DIR, "non_chitinase_dedup_prefpfamSI.tsv")

# ------------------ checks & prep ------------------
need_cols = ["protein_accession","analysis","signature_accession","start","end","score"]
for c in need_cols:
    if c not in non_chitinase_rows.columns:
        raise KeyError(f"Required column '{c}' missing in non_chitinase_rows")

df = non_chitinase_rows.copy()

# numeric coercions
df["start"] = pd.to_numeric(df["start"], errors="coerce")
df["end"]   = pd.to_numeric(df["end"], errors="coerce")
df["score_num"] = pd.to_numeric(df["score"], errors="coerce")
df = df.dropna(subset=["protein_accession","analysis","signature_accession","start","end"]).copy()

df["start"] = df["start"].astype(int)
df["end"]   = df["end"].astype(int)
df["span"]  = (df["end"] - df["start"] + 1).clip(lower=0)

# global frequency of signature_accession (use existing sig_freq if present)
if "sig_freq" in globals():
    if isinstance(sig_freq, pd.DataFrame):
        if not {"signature_accession","count"}.issubset(sig_freq.columns):
            raise KeyError("sig_freq DataFrame must have ['signature_accession','count']")
        sig_freq_map = dict(zip(sig_freq["signature_accession"], sig_freq["count"]))
    elif isinstance(sig_freq, dict):
        sig_freq_map = sig_freq
    else:
        raise TypeError("sig_freq exists but is neither DataFrame nor dict.")
else:
    sig_freq_map = df["signature_accession"].value_counts(dropna=False).to_dict()

df["sig_freq_global"] = df["signature_accession"].map(sig_freq_map).fillna(0).astype(int)

# ------------------ overlap logic ------------------
def overlap_frac(a_start, a_end, b_start, b_end):
    inter = max(0, min(a_end, b_end) - max(a_start, b_start) + 1)
    shorter = max(1, min(a_end - a_start + 1, b_end - b_start + 1))
    return inter / shorter

def is_same_group(a_start, a_end, b_start, b_end):
    # either both ends within tolerance OR overlap fraction >= threshold
    close_ends = (abs(a_start - b_start) <= POS_TOL) and (abs(a_end - b_end) <= POS_TOL)
    sig_overlap = overlap_frac(a_start, a_end, b_start, b_end) >= OVERLAP_FRAC
    return close_ends or sig_overlap

def choose_rep(group_df: pd.DataFrame) -> pd.Series:
    # 1) Prefer Pfam rows (case-insensitive)
    pfam_mask = group_df["analysis"].str.lower().eq("pfam")
    if pfam_mask.any():
        cand = group_df.loc[pfam_mask].copy()
    else:
        cand = group_df.copy()

    # 2) Otherwise (or within Pfam ties) pick highest global frequency,
    #    3) then longer span, 4) lower score, 5) earlier start
    cand = cand.sort_values(
        by=["sig_freq_global","span","score_num","start"],
        ascending=[False, False, True, True]
    )
    return cand.iloc[0]

def dedup_one_protein(sub: pd.DataFrame) -> pd.DataFrame:
    # Build graph of overlapping intervals across ALL identifiers
    sub = sub.reset_index(drop=True)
    n = len(sub)
    if n <= 1:
        return sub

    starts = sub["start"].to_numpy()
    ends   = sub["end"].to_numpy()
    adj = [[] for _ in range(n)]

    for i in range(n):
        for j in range(i+1, n):
            if is_same_group(starts[i], ends[i], starts[j], ends[j]):
                adj[i].append(j)
                adj[j].append(i)

    # Connected components -> one representative per component
    seen = [False]*n
    kept_rows = []
    for i in range(n):
        if seen[i]:
            continue
        comp_idx = []
        q = deque([i])
        seen[i] = True
        while q:
            k = q.popleft()
            comp_idx.append(k)
            for nb in adj[k]:
                if not seen[nb]:
                    seen[nb] = True
                    q.append(nb)

        comp = sub.iloc[comp_idx].copy()
        rep = choose_rep(comp)
        kept_rows.append(rep)

    return pd.DataFrame(kept_rows)

# ------------------ apply per protein ------------------
dedup_pieces = []
for prot, sub in df.groupby("protein_accession", sort=False):
    dedup_pieces.append(dedup_one_protein(sub))

non_chitinase_dedup_prefpfam = (
    pd.concat(dedup_pieces, ignore_index=True) if dedup_pieces else df.iloc[0:0].copy()
)

# save & preview
non_chitinase_dedup_prefpfam.to_csv(OUT_PATH, sep="\t", index=False)
print(f"✓ Wrote deduplicated table (prefer Pfam): {OUT_PATH}")
print(f"  Proteins covered: {non_chitinase_dedup_prefpfam['protein_accession'].nunique()}")
print(f"  Rows kept:        {len(non_chitinase_dedup_prefpfam)}")

# Inspect your example protein to verify multiple distinct domains survive
example = "M2PU94"
if example in set(non_chitinase_dedup_prefpfam["protein_accession"]):
    display(
        non_chitinase_dedup_prefpfam.loc[
            non_chitinase_dedup_prefpfam["protein_accession"] == example,
            ["protein_accession","analysis","signature_accession","interpro_accession",
             "signature_description","interpro_description","start","end","score","sig_freq_global"]
        ].sort_values(["start","end"])
    )
else:
    display(non_chitinase_dedup_prefpfam.head(12))


✓ Wrote deduplicated table (prefer Pfam): SI/non_chitinase_dedup_prefpfamSI.tsv
  Proteins covered: 28254
  Rows kept:        63644


,protein_accession,analysis,signature_accession,interpro_accession,signature_description,interpro_description,start,end,score,sig_freq_global
2,M2PU94,Pfam,PF02018,IPR003305,Carbohydrate binding domain,"Carbohydrate-binding, CenC-like",30,139,3.000000e-09,2612
3,M2PU94,Pfam,PF00041,IPR003961,Fibronectin type III domain,Fibronectin type III,165,242,2.600000e-06,6556


In [22]:
# --- Fill NaNs in base columns from any *_x, *_y duplicate columns ------------
import re
import pandas as pd

# df is your working DataFrame (e.g., df = alldom_anno_dedupe_clean.copy())
# If your DataFrame variable is different, replace 'df' below.

def fill_from_dupes(df, drop_dupes=True):
    cols = list(df.columns)

    # find all *_x / *_y columns
    dup_map = {}  # base -> [dup cols]
    for c in cols:
        m = re.match(r"^(.*)_(x|y)$", c)
        if m:
            base = m.group(1)
            dup_map.setdefault(base, []).append(c)

    report = []
    for base, dups in dup_map.items():
        # ensure base column exists; if not, create as all-NaN (object dtype)
        if base not in df.columns:
            df[base] = pd.Series([pd.NA]*len(df), dtype="object")

        before_missing = df[base].isna().sum()

        # fill from each dup in a stable order (_x then _y)
        for dup in sorted(dups):  # alphabetical order: _x first, then _y
            df[base] = df[base].fillna(df[dup])

        after_missing = df[base].isna().sum()
        filled = before_missing - after_missing
        report.append((base, filled, dups))

        # optionally drop dup columns to tidy up
        if drop_dupes:
            df.drop(columns=dups, inplace=True, errors="ignore")

    # summary
    if report:
        print("✅ Filled NaNs from duplicate columns:")
        for base, filled, dups in sorted(report, key=lambda x: x[0]):
            print(f"  - {base}: filled {filled} from {dups}")
    else:
        print("ℹ️ No *_x / *_y duplicate columns found.")

    return df

# run it
df = fill_from_dupes(parsed_chitinase_dom, drop_dupes=True)

# peek
df


ℹ️ No *_x / *_y duplicate columns found.


,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,...,sequence,sequence_len_fasta,is_chitinase_related,matched_terms,start_num,end_num,score_num,span,priority_rank,sig_freq_global
0,A0A010YG22,BCAC65E13E44ACC0D5889A40D5B1A2F8,354,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,354,1.020352e+01,T,...,MARGQRRRNEKGRWVLIGVAVVAVIAAAATAVVFLRDGDDGKDGTV...,354,True,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,354,1.020352e+01,299,2.0,39036
1,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF00704,Glycosyl hydrolases family 18,137,428,2.300000e-16,T,...,MSQPPFLDPAPAPGASPKTAKAVFFDPANKRWPRLRLGMALIGVAL...,1156,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,137,428,2.300000e-16,292,1.0,36361
2,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,503,716,1.200000e-57,T,...,MSQPPFLDPAPAPGASPKTAKAVFFDPANKRWPRLRLGMALIGVAL...,1156,True,glycoside hydrolase,503,716,1.200000e-57,214,NaN,1399
3,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,Pfam,PF00704,Glycosyl hydrolases family 18,136,428,2.200000e-16,T,...,MSQSLSGDPQPRPGSGPKIAAAVFFDPANKRWPRLRLGMALIAVTL...,1156,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,136,428,2.200000e-16,293,1.0,36361
4,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,504,716,3.400000e-57,T,...,MSQSLSGDPQPRPGSGPKIAAAVFFDPANKRWPRLRLGMALIAVTL...,1156,True,glycoside hydrolase,504,716,3.400000e-57,213,NaN,1399
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41899,X6Q4Q0,83C4D5DBFD8A1179D0BA7A5552A33164,1132,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,488,704,6.800000e-55,T,...,MSKQVFQTDSRQRWSYFKWTLRVILTILSLLGIVFLAMFALEGSPQ...,1132,True,glycoside hydrolase,488,704,6.800000e-55,217,NaN,1399
41900,X8GPF2,70D3A80C4982E83860FD2DA40F7823CA,567,Pfam,PF00704,Glycosyl hydrolases family 18,334,554,2.100000e-20,T,...,MKKKKKRIPLALILLLVIVLLGIVIFLVVRYRPGKEHLSLNTFFGV...,567,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,334,554,2.100000e-20,221,1.0,36361
41901,X8HE75,A456F3A5EAF449088DBE76505D69FC5A,539,Pfam,PF00704,Glycosyl hydrolases family 18,206,422,2.900000e-19,T,...,MLRKFAMVTTLAAAVLSSISGNTVQAEVLVPLQQYLNTTNRNYEAN...,539,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,206,422,2.900000e-19,217,1.0,36361
41902,Z9JN00,A28869D690FF03DAD30FDC1D4D83035F,351,Pfam,PF00704,Glycosyl hydrolases family 18,37,336,1.000000e-21,T,...,MTKRSLALLLTLIFAVPAAAKNPTALFYLMGTQKSTNSFLAHADKI...,351,True,glycoside hydrolase;glycosyl hydrolase;ipr0012...,37,336,1.000000e-21,300,1.0,36361


In [23]:
# --- Add per-protein domain counts and order numbers --------------------------

# Count total number of domains per protein_accession
df["Total_num_GH18domains"] = df.groupby("protein_accession")["protein_accession"].transform("count")

# Assign sequential domain number (1, 2, 3, …) per protein_accession
df["GH18domain_num"] = df.groupby("protein_accession").cumcount() + 1

# --- Extract GH18 sequence for each domain ------------------------------------
def extract_domain(seq, start, end):
    """Slice subsequence from start–end (1-based, inclusive)."""
    if not isinstance(seq, str) or pd.isna(start) or pd.isna(end):
        return pd.NA
    try:
        s, e = int(start), int(end)
        if s < 1 or e > len(seq) or s > e:
            return pd.NA
        return seq[s-1:e]
    except Exception:
        return pd.NA

df["GH18sequence"] = df.apply(
    lambda r: extract_domain(r["sequence"], r["start"], r["end"]),
    axis=1
)

print("✅ Added columns:")
print("  - Total_num_GH18domains: total domains per protein_accession")
print("  - GH18domain_num: position of domain (ordered by start)")
print("  - GH18sequence: extracted subsequence based on start/end")

# Quick summary
summary = (
    df.groupby("Total_num_GH18domains")
    .size()
    .reset_index(name="num_proteins")
    .sort_values("Total_num_GH18domains")
)
print("\nDistribution of domain counts:")
display(summary)

# Peek
display(df.head(10))


✅ Added columns:
  - Total_num_GH18domains: total domains per protein_accession
  - GH18domain_num: position of domain (ordered by start)
  - GH18sequence: extracted subsequence based on start/end

Distribution of domain counts:


,Total_num_GH18domains,num_proteins
0,1,37321
1,2,4400
2,3,183


,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,...,matched_terms,start_num,end_num,score_num,span,priority_rank,sig_freq_global,Total_num_GH18domains,GH18domain_num,GH18sequence
0,A0A010YG22,BCAC65E13E44ACC0D5889A40D5B1A2F8,354,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,354,1.020352e+01,T,...,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,354,1.020352e+01,299,2.0,39036,1,1,YAPYVYVTLADRPSLTEIAQKTGANGLHLGFAITKGDECDLTWDGT...
1,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF00704,Glycosyl hydrolases family 18,137,428,2.300000e-16,T,...,glycoside hydrolase;glycosyl hydrolase;ipr0012...,137,428,2.300000e-16,292,1.0,36361,2,1,AFFVNWDDASMSSLKENLGSLDTVIAEWLHLASEDGTLRENDPLRT...
2,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,503,716,1.200000e-57,T,...,glycoside hydrolase,503,716,1.200000e-57,214,NaN,1399,2,2,PYVITRHGSAERKVVLTFDDGPDPRFTPLILDALRDAGVPATFFII...
3,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,Pfam,PF00704,Glycosyl hydrolases family 18,136,428,2.200000e-16,T,...,glycoside hydrolase;glycosyl hydrolase;ipr0012...,136,428,2.200000e-16,293,1.0,36361,2,1,IGFFVNWDDASMSSLKENLGSLDTVIAEWLHLASEDGTLRENDPLR...
4,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,504,716,3.400000e-57,T,...,glycoside hydrolase,504,716,3.400000e-57,213,NaN,1399,2,2,YVITRHGGAERKIVLTFDDGPDPRFTPLVLDALRDAGVPATFFIIG...
5,A0A014LJM1,8495312E26224E3E1925B2F1461C43E6,344,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,344,1.060503e+01,T,...,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,344,1.060503e+01,289,2.0,39036,1,1,AAVAPYLYNGWGNPPDPGEVMDATGVGWFTLAFVLDAGNCDPRWDG...
6,A0A014LR51,9F864E95FB4997720FE7F921ECFB6456,336,Pfam,PF00704,Glycosyl hydrolases family 18,51,270,3.600000e-14,T,...,glycoside hydrolase;glycosyl hydrolase;ipr0012...,51,270,3.600000e-14,220,1.0,36361,1,1,VTGYWQNFDNGATKQKLSDVPDDYDIIAVAFADATGTPGAVDFKLD...
7,A0A014LVP2,44CFF99BCD9A69393EB542696BBF9637,311,Pfam,PF00704,Glycosyl hydrolases family 18,80,253,3.800000e-08,T,...,glycoside hydrolase;glycosyl hydrolase;ipr0012...,80,253,3.800000e-08,174,1.0,36361,1,1,VAVIFAANINYDTGTKAAYLHFNENVQRVLDNAATEIRPLQQKGIK...
8,A0A014M4I2,26C57647ACF0EF3CEFA8F8E9E6E5C545,421,Pfam,PF00704,Glycosyl hydrolases family 18,57,410,6.800000e-72,T,...,glycoside hydrolase;glycosyl hydrolase;ipr0012...,57,410,6.800000e-72,354,1.0,36361,1,1,KVVGYFTEWGIYQRNYHVKNVETSGSAAKLTHINYAFGNVTGGKCA...
9,A0A014MIZ7,F369CDD9CFC7DB25C25543325A21CB1C,350,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,341,9.859364e+00,T,...,gh18;glycoside hydrolase;glycosyl hydrolase;ip...,56,341,9.859364e+00,286,2.0,39036,1,1,GGFAPYIDTSLYPAYDMVGTADKTGVKEFNLAFITSGGGCTPKWGG...


In [24]:
#remove columns Total_num_domains and domain_num
df["fastaID"] = df["protein_accession"]+"|"+df["GH18domain_num"].astype(str)+"of"+df["Total_num_GH18domains"].astype(str)
df

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,...,start_num,end_num,score_num,span,priority_rank,sig_freq_global,Total_num_GH18domains,GH18domain_num,GH18sequence,fastaID
0,A0A010YG22,BCAC65E13E44ACC0D5889A40D5B1A2F8,354,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,354,1.020352e+01,T,...,56,354,1.020352e+01,299,2.0,39036,1,1,YAPYVYVTLADRPSLTEIAQKTGANGLHLGFAITKGDECDLTWDGT...,A0A010YG22|1of1
1,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF00704,Glycosyl hydrolases family 18,137,428,2.300000e-16,T,...,137,428,2.300000e-16,292,1.0,36361,2,1,AFFVNWDDASMSSLKENLGSLDTVIAEWLHLASEDGTLRENDPLRT...,A0A011PM39|1of2
2,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,503,716,1.200000e-57,T,...,503,716,1.200000e-57,214,NaN,1399,2,2,PYVITRHGSAERKVVLTFDDGPDPRFTPLILDALRDAGVPATFFII...,A0A011PM39|2of2
3,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,Pfam,PF00704,Glycosyl hydrolases family 18,136,428,2.200000e-16,T,...,136,428,2.200000e-16,293,1.0,36361,2,1,IGFFVNWDDASMSSLKENLGSLDTVIAEWLHLASEDGTLRENDPLR...,A0A011PXQ4|1of2
4,A0A011PXQ4,BCAE3BD21D067344C874DD0EB6344E16,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,504,716,3.400000e-57,T,...,504,716,3.400000e-57,213,NaN,1399,2,2,YVITRHGGAERKIVLTFDDGPDPRFTPLVLDALRDAGVPATFFIIG...,A0A011PXQ4|2of2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41899,X6Q4Q0,83C4D5DBFD8A1179D0BA7A5552A33164,1132,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,488,704,6.800000e-55,T,...,488,704,6.800000e-55,217,NaN,1399,2,2,TYTIQRLGECKDKQLVITFDDGPDSRWTPTVLSTLKKYNVPAAFFM...,X6Q4Q0|2of2
41900,X8GPF2,70D3A80C4982E83860FD2DA40F7823CA,567,Pfam,PF00704,Glycosyl hydrolases family 18,334,554,2.100000e-20,T,...,334,554,2.100000e-20,221,1.0,36361,1,1,KTVLNTTSARDNLVNGLIAAAIRYNLDGINVDIESLPTEAADGYIQ...,X8GPF2|1of1
41901,X8HE75,A456F3A5EAF449088DBE76505D69FC5A,539,Pfam,PF00704,Glycosyl hydrolases family 18,206,422,2.900000e-19,T,...,206,422,2.900000e-19,217,1.0,36361,1,1,ILADEAVWKKYAQNLIQYAYIYGFDGYNFDFENVDYSDRDKLTRFV...,X8HE75|1of1
41902,Z9JN00,A28869D690FF03DAD30FDC1D4D83035F,351,Pfam,PF00704,Glycosyl hydrolases family 18,37,336,1.000000e-21,T,...,37,336,1.000000e-21,300,1.0,36361,1,1,NSFLAHADKIDTLVPTWYSVDPKGLVNGAPNERIYRIAQKKKITVT...,Z9JN00|1of1


In [25]:
df.to_csv('SI/gh18alldomains_multiperseq_numdomainsannot_SI.csv')

In [29]:
# 5) write FASTA with protein_accession and GH18 sequence
FASTA_OUT = os.path.join(OUT_DIR, "pfamscan_gh18domain_seq_20251127.fasta")

with open(FASTA_OUT, "w") as fh:
    for _, r in df.iterrows():
        acc = r["fastaID"]
        seq = r["GH18sequence"]
        fh.write(f">{acc}\n")
        for i in range(0, len(seq), 60):
            fh.write(seq[i:i+60] + "\n")

print(f"✓ Wrote FASTA: {FASTA_OUT}")


✓ Wrote FASTA: SI/pfamscan_gh18domain_seq_20251127.fasta


In [31]:
df['GH18_ID'] = 'True'
alldom_anno_dedupe = pd.concat([df, non_chitinase_dedup_prefpfam], ignore_index=True)
#order by protein_accession and start
alldom_anno_dedupe = alldom_anno_dedupe.sort_values(by=["protein_accession", "start"], ascending=[True, True])
alldom_anno_dedupe

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,end,score,status,...,end_num,score_num,span,priority_rank,sig_freq_global,Total_num_GH18domains,GH18domain_num,GH18sequence,fastaID,GH18_ID
0,A0A010YG22,BCAC65E13E44ACC0D5889A40D5B1A2F8,354,PROSITE profiles,PS51910,Glycosyl hydrolases family 18 (GH18) domain pr...,56,354,1.020352e+01,T,...,354.0,1.020352e+01,299,2.0,39036,1.0,1.0,YAPYVYVTLADRPSLTEIAQKTGANGLHLGFAITKGDECDLTWDGT...,A0A010YG22|1of1,True
1,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF00704,Glycosyl hydrolases family 18,137,428,2.300000e-16,T,...,428.0,2.300000e-16,292,1.0,36361,2.0,1.0,AFFVNWDDASMSSLKENLGSLDTVIAEWLHLASEDGTLRENDPLRT...,A0A011PM39|1of2,True
2,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,CATH-Gene3D,G3DSA:3.20.20.370,Glycoside hydrolase/deacetylase,503,716,1.200000e-57,T,...,716.0,1.200000e-57,214,NaN,1399,2.0,2.0,PYVITRHGSAERKVVLTFDDGPDPRFTPLILDALRDAGVPATFFII...,A0A011PM39|2of2,True
99331,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,PIRSR,PIRSR037479-3,NaN,513,711,8.700000e-44,T,...,NaN,8.700000e-44,199,NaN,1396,NaN,NaN,NaN,NaN,NaN
99330,A0A011PM39,2B03A5B39A2C5137BCACBC4D01697D06,1156,Pfam,PF13641,Glycosyltransferase like family 2,785,1008,4.500000e-30,T,...,NaN,4.500000e-30,224,NaN,622,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41901,X8HE75,A456F3A5EAF449088DBE76505D69FC5A,539,Pfam,PF00704,Glycosyl hydrolases family 18,206,422,2.900000e-19,T,...,422.0,2.900000e-19,217,1.0,36361,1.0,1.0,ILADEAVWKKYAQNLIQYAYIYGFDGYNFDFENVDYSDRDKLTRFV...,X8HE75|1of1,True
69375,X8HE75,A456F3A5EAF449088DBE76505D69FC5A,539,MobiDB-lite,mobidb-lite,Consensus disorder prediction,480,520,NaN,T,...,NaN,NaN,41,NaN,25635,NaN,NaN,NaN,NaN,NaN
41902,Z9JN00,A28869D690FF03DAD30FDC1D4D83035F,351,Pfam,PF00704,Glycosyl hydrolases family 18,37,336,1.000000e-21,T,...,336.0,1.000000e-21,300,1.0,36361,1.0,1.0,NSFLAHADKIDTLVPTWYSVDPKGLVNGAPNERIYRIAQKKKITVT...,Z9JN00|1of1,True
44047,Z9JR83,F8D2A83931EDBBF7DF01F4D4CCBE7BE6,316,PROSITE profiles,PS51318,Twin arginine translocation (Tat) signal profile,1,37,8.472891e+00,T,...,NaN,8.472891e+00,37,NaN,1851,NaN,NaN,NaN,NaN,NaN


In [ ]:
#remove rows with COILS or MobiDB-lite in analysis column
alldom_anno_dedupe_clean = alldom_anno_dedupe[~alldom_anno_dedupe["analysis"].isin(["COILS", "MobiDB-lite","PIRSR"])].copy()
alldom_anno_dedupe_clean.reset_index(drop=True, inplace=True)
alldom_anno_dedupe_clean.to_csv('SI/all_domains_annotated_deduplicated_cleaned_noPIRSR.tsv', sep='\t', index=False)

generating domain signatures

In [280]:
# --- Remove domains shorter than 100 amino acids ---

# Make sure start and end are numeric
alldom_anno_dedupe_clean["start"] = pd.to_numeric(alldom_anno_dedupe_clean["start"], errors="coerce")
alldom_anno_dedupe_clean["end"]   = pd.to_numeric(alldom_anno_dedupe_clean["end"], errors="coerce")

# Compute domain length
alldom_anno_dedupe_clean["domain_length"] = (alldom_anno_dedupe_clean["end"] - alldom_anno_dedupe_clean["start"] + 1)

# Filter out domains shorter than 100 aa
before = len(alldom_anno_dedupe_clean)
alldom_anno_dedupe_clean = alldom_anno_dedupe_clean[alldom_anno_dedupe_clean["domain_length"] >= 70].copy()
after = len(alldom_anno_dedupe_clean)

alldom_anno_dedupe_clean.to_csv('all_domains_annotated_deduplicated_cleaned_greater70AA.tsv', sep='\t', index=False)

print(f"✅ Removed {before - after} domains shorter than 100 amino acids.")
print(f"Remaining rows: {after}")



✅ Removed 27568 domains shorter than 100 amino acids.
Remaining rows: 68291


In [360]:
import pandas as pd
import numpy as np
from typing import Optional

tmp = pd.read_csv("all_domains_annotated_deduplicated_cleaned.tsv", sep="\t")

# numeric start used for ordering (prefer start_num if present)
tmp["__start_sort__"] = pd.to_numeric(
    tmp["start_num"] if "start_num" in tmp else tmp["start"],
    errors="coerce"
)



def best_seq_len(g: pd.DataFrame) -> Optional[int]:
    a = pd.to_numeric(g.get("sequence_len_fasta"), errors="coerce")
    if a.notna().any():
        return int(a.dropna().iloc[0])
    b = pd.to_numeric(g.get("sequence_length"), errors="coerce")
    if b.notna().any():
        return int(b.dropna().iloc[0])
    c = g.get("sequence")
    if c is not None and c.notna().any():
        return int(c.dropna().str.len().iloc[0])
    return None

def build_rows(df: pd.DataFrame) -> list[dict]:
    rows = []
    for prot, g in df.groupby("protein_accession", sort=False):
        g = g.sort_values("__start_sort__", kind="mergesort")

        # fingerprint: all signature_accession in order (keep duplicates)
        sigs = g["signature_accession"].dropna().astype(str).tolist()
        fingerprint = " ".join(sigs)

        # spans: accession[ start-end ]
        s = pd.to_numeric(g["start"], errors="coerce")
        e = pd.to_numeric(g["end"],   errors="coerce")
        ok = s.notna() & e.notna() & g["signature_accession"].notna()
        g2 = g.loc[ok, ["signature_accession"]].copy()
        g2["start_i"] = s[ok].astype(int)
        g2["end_i"]   = e[ok].astype(int)
        spans = " ".join(f"{sig}[{st}-{en}]"
                         for sig, st, en in zip(g2["signature_accession"], g2["start_i"], g2["end_i"]))

        rows.append({
            "protein_accession": prot,
            "signature_accession_fingerprint": fingerprint,
            "signature_spans": spans,
            "sequence_length_total": best_seq_len(g),
        })
    return rows

protein_fingerprints = pd.DataFrame(build_rows(tmp))

# --- Add GH18_ID column to protein_fingerprints ----------------------------

# Subset GH18-positive rows
gh18_hits = alldom_anno_dedupe_clean.loc[
    alldom_anno_dedupe_clean["GH18_ID"].notna()
    | (alldom_anno_dedupe_clean["GH18_ID"] == True)
].copy()

# Collect GH18 signature_accessions per protein_accession
gh18_map = (
    gh18_hits.groupby("protein_accession")["signature_accession"]
    .apply(lambda x: " ".join(sorted(set(x.dropna().astype(str)))))
    .to_dict()
)

# Map onto protein_fingerprints
protein_fingerprints["GH18_ID"] = protein_fingerprints["protein_accession"].map(gh18_map)

# Replace empty strings with NaN for clarity
protein_fingerprints["GH18_ID"].replace("", np.nan, inplace=True)

print(f"✅ Added GH18_ID column to protein_fingerprints ({protein_fingerprints['GH18_ID'].notna().sum()} proteins with GH18 domains)")
display(protein_fingerprints.head(10))
# Optional save:
protein_fingerprints.to_csv("all_domains_annotated_deduplicated_cleaned_proteinfprint.tsv", sep="\t", index=False)


✅ Added GH18_ID column to protein_fingerprints (39582 proteins with GH18 domains)


,protein_accession,signature_accession_fingerprint,signature_spans,sequence_length_total,GH18_ID
0,A0A010YG22,PS51910,PS51910[56-354],354,PS51910
1,A0A011PM39,PF00704 G3DSA:3.20.20.370 PF01522 PF13641,PF00704[137-428] G3DSA:3.20.20.370[503-716] PF...,1156,G3DSA:3.20.20.370 PF00704
2,A0A011PXQ4,PF00704 G3DSA:3.20.20.370 PF01522 PF13641,PF00704[136-428] G3DSA:3.20.20.370[504-716] PF...,1156,G3DSA:3.20.20.370 PF00704
3,A0A014LJM1,PS51910 PS51318,PS51910[56-344] PS51318[1-41],344,PS51910
4,A0A014LR51,PF00704,PF00704[51-270],336,PF00704
5,A0A014LVP2,PF00704 cd06542,PF00704[80-253] cd06542[49-303],311,PF00704
6,A0A014M4I2,PF00704,PF00704[57-410],421,PF00704
7,A0A014MIZ7,PS51910,PS51910[56-341],350,PS51910
8,A0A014MM96,PF00704 PF02018 PF00041,PF00704[283-506] PF02018[46-162] PF00041[189-265],570,PF00704
9,A0A014MSN8,PS51910 PS51318,PS51910[45-336] PS51318[1-42],336,PS51910


In [362]:
annot = pd.read_csv('../../250710/downloads/allipr001223_bact_arch_jackseq_annot.tsv', sep='\t')
#merge the Organism column from annot to protein_fingerprints based on protein_accession
protein_fingerprints_org = protein_fingerprints.merge(annot[['Entry', 'Organism', 'Organism (ID)']], left_on='protein_accession', right_on='Entry',how='left')
#are ther any NaNs in the Organism column
protein_fingerprints_org#[protein_fingerprints_org['Organism'].isna()]

/var/folders/k3/7_cndd213s1_cf9yqdgvytf80000gn/T/ipykernel_57392/1800871430.py:1: DtypeWarning: Columns (12,15,21) have mixed types. Specify dtype option on import or set low_memory=False.
  annot = pd.read_csv('../../250710/downloads/allipr001223_bact_arch_jackseq_annot.tsv', sep='\t')


,protein_accession,signature_accession_fingerprint,signature_spans,sequence_length_total,GH18_ID,signature_accession_fingerprint_GH18replaced,num_of_domains,no_GH18_domains,Entry,Organism,Organism (ID)
0,A0A010YG22,PS51910,PS51910[56-354],354,PS51910,GH18,1,1,A0A010YG22,Cryptosporangium arvum DSM 44712,927661
1,A0A011PM39,PF00704 G3DSA:3.20.20.370 PF01522 PF13641,PF00704[137-428] G3DSA:3.20.20.370[503-716] PF...,1156,G3DSA:3.20.20.370 PF00704,GH18 GH18 PF01522 PF13641,4,2,A0A011PM39,Accumulibacter regalis,522306
2,A0A011PXQ4,PF00704 G3DSA:3.20.20.370 PF01522 PF13641,PF00704[136-428] G3DSA:3.20.20.370[504-716] PF...,1156,G3DSA:3.20.20.370 PF00704,GH18 GH18 PF01522 PF13641,4,2,A0A011PXQ4,Candidatus Accumulibacter appositus,1454003
3,A0A014LJM1,PS51910 PS51318,PS51910[56-344] PS51318[1-41],344,PS51910,GH18 PS51318,2,1,A0A014LJM1,Streptomyces sp. PRh5,1158056
4,A0A014LR51,PF00704,PF00704[51-270],336,PF00704,GH18,1,1,A0A014LR51,Streptomyces sp. PRh5,1158056
...,...,...,...,...,...,...,...,...,...,...,...
39577,X6Q4Q0,PF00704 G3DSA:3.20.20.370 PF01522 PF00535,PF00704[196-410] G3DSA:3.20.20.370[488-704] PF...,1132,G3DSA:3.20.20.370 PF00704,GH18 GH18 PF01522 PF00535,4,2,X6Q4Q0,Prevotella sp. ICM33,1161412
39578,X8GPF2,PF00704 G3DSA:2.30.30.40,PF00704[334-554] G3DSA:2.30.30.40[169-234],567,PF00704,GH18 G3DSA:2.30.30.40,2,1,X8GPF2,Shuttleworthella sp. MSX8B,936574
39579,X8HE75,PF00704,PF00704[206-422],539,PF00704,GH18,1,1,X8HE75,Veillonella sp. ICM51a,936591
39580,Z9JN00,PF00704,PF00704[37-336],351,PF00704,GH18,1,1,Z9JN00,Xylella taiwanensis,1444770


In [325]:
#replace all GH18 signatures with 'GH18' in the signature_accession_fingerprint column but leave non GH18 signatures as is to create a GH18_fingerprint column
#df['GH18_fingerprint'] = df['signature_accession_fingerprint'].apply(lambda x: 'GH18' if 'GH18' in x else x)
protein_fingerprints_org['GH18_fingerprint'] = protein_fingerprints_org['signature_accession_fingerprint'].apply(
    lambda x: ' '.join(['GH18' if sig in set().union(*[s.split() for s in protein_fingerprints_org['GH18_ID'].dropna().unique()]) else sig for sig in x.split()])
)
protein_fingerprints_org

,protein_accession,signature_accession_fingerprint,signature_spans,sequence_length_total,GH18_ID,signature_accession_fingerprint_GH18replaced,num_of_domains,no_GH18_domains,Entry,Organism,Organism (ID),non_GH18_fingerprint,GH18_fingerprint
0,A0A010YG22,PS51910,PS51910[56-354],354,PS51910,GH18,1,1,A0A010YG22,Cryptosporangium arvum DSM 44712,927661,,GH18
1,A0A011PM39,PF00704 G3DSA:3.20.20.370 PF01522 PF13641,PF00704[137-428] G3DSA:3.20.20.370[503-716] PF...,1156,G3DSA:3.20.20.370 PF00704,GH18 GH18 PF01522 PF13641,4,2,A0A011PM39,Accumulibacter regalis,522306,PF01522 PF13641,GH18 GH18 PF01522 PF13641
2,A0A011PXQ4,PF00704 G3DSA:3.20.20.370 PF01522 PF13641,PF00704[136-428] G3DSA:3.20.20.370[504-716] PF...,1156,G3DSA:3.20.20.370 PF00704,GH18 GH18 PF01522 PF13641,4,2,A0A011PXQ4,Candidatus Accumulibacter appositus,1454003,PF01522 PF13641,GH18 GH18 PF01522 PF13641
3,A0A014LJM1,PS51910 PS51318,PS51910[56-344] PS51318[1-41],344,PS51910,GH18 PS51318,2,1,A0A014LJM1,Streptomyces sp. PRh5,1158056,PS51318,GH18 PS51318
4,A0A014LR51,PF00704,PF00704[51-270],336,PF00704,GH18,1,1,A0A014LR51,Streptomyces sp. PRh5,1158056,,GH18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39577,X6Q4Q0,PF00704 G3DSA:3.20.20.370 PF01522 PF00535,PF00704[196-410] G3DSA:3.20.20.370[488-704] PF...,1132,G3DSA:3.20.20.370 PF00704,GH18 GH18 PF01522 PF00535,4,2,X6Q4Q0,Prevotella sp. ICM33,1161412,PF01522 PF00535,GH18 GH18 PF01522 PF00535
39578,X8GPF2,PF00704 G3DSA:2.30.30.40,PF00704[334-554] G3DSA:2.30.30.40[169-234],567,PF00704,GH18 G3DSA:2.30.30.40,2,1,X8GPF2,Shuttleworthella sp. MSX8B,936574,G3DSA:2.30.30.40,GH18 G3DSA:2.30.30.40
39579,X8HE75,PF00704,PF00704[206-422],539,PF00704,GH18,1,1,X8HE75,Veillonella sp. ICM51a,936591,,GH18
39580,Z9JN00,PF00704,PF00704[37-336],351,PF00704,GH18,1,1,Z9JN00,Xylella taiwanensis,1444770,,GH18


In [363]:
protein_fingerprints_org.to_csv("all_domains_annotated_deduplicated_cleaned_proteinfprint_withorg.tsv", sep="\t", index=False)

In [270]:
#non_gh18_fingerprint dataframe with columns protein_accession, Organism, non_GH18_fingerprint
non_gh18_fingerprint = protein_fingerprints_org[['protein_accession', 'Organism', 'signature_accession_fingerprint']].copy()
#remove rows with empty non_GH18_fingerprint
#non_gh18_fingerprint = non_gh18_fingerprint[non_gh18_fingerprint['non_GH18_fingerprint'] != '']
#reset index
non_gh18_fingerprint.reset_index(drop=True, inplace=True)
non_gh18_fingerprint.to_csv('protein_unedited_fingerprints.tsv', sep='\t', index=False)
print("✓ Wrote protein_GH18_fingerprints.tsv")

✓ Wrote protein_GH18_fingerprints.tsv
